# Notebook 2: Hydrological Loading & Surface Load Effects on dv/v## Companion to: *Monitoring Velocity Change Over 20 Years at Parkfield* (Okubo et al., 2024)This notebook models pore-pressure diffusion, groundwater-level effects, and surface loading (rain, ice, snow) on seismic velocity changes using poroelastic theory.### Key References- Fokker, E., et al. (2021). Physics-based relationship for pore pressure and vertical stress monitoring using seismic velocity variations. *Remote Sensing*, 13, 2684.- Clements, T. & Denolle, M. A. (2023). The seismic signature of California's earthquakes, droughts, and floods. *JGR Solid Earth*.- Roeloffs, E. (1988). Fault stability changes induced beneath a reservoir. *JGR*, 93, 2107–2124.- Talwani, P., Chen, L., & Gahalaut, K. (2007). Seismogenic permeability. *JGR*, 112, B07309.- Okubo, K., Delbridge, B. G., & Denolle, M. A. (2024). Monitoring velocity change over 20 years at Parkfield. *JGR Solid Earth*.

In [ ]:
import numpy as npimport matplotlib.pyplot as pltfrom scipy.special import erf, erfcfrom matplotlib.colors import Normalizeplt.rcParams.update({    'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 14,    'xtick.labelsize': 11, 'ytick.labelsize': 11,    'figure.dpi': 150, 'font.family': 'serif', 'mathtext.fontset': 'cm',    'axes.grid': True, 'grid.alpha': 0.3})print("Environment ready.")

## 1. Pore Pressure Diffusion Model (Roeloffs, 1988)The coupled poroelastic response at depth $z$ to a surface load $p_0$ is (Roeloffs, 1988):$$P(z, t) = \frac{B(1+\nu_u)}{3(1-\nu_u)} p_0 \operatorname{erf}\left[\frac{z}{(4ct)^{1/2}}\right] + p_0 \operatorname{erfc}\left[\frac{z}{(4ct)^{1/2}}\right]$$- **First term**: undrained poroelastic response (elastic loading)- **Second term**: drained response (pore pressure diffusion)The **Skempton coefficient** $B$ and **undrained Poisson's ratio** $\nu_u$ control the partitioning.

In [ ]:
# === Pore Pressure Diffusion (Roeloffs, 1988) ===def pore_pressure_roeloffs(z, t, p0, c, B=0.8, nu_u=0.35):    """    Coupled poroelastic pore pressure response at depth z, time t.    Roeloffs (1988); Clements & Denolle (2023) Eq. 8.        Parameters:    -----------    z : depth [m]    t : time since load [s]    p0 : surface load pressure [Pa]    c : hydraulic diffusivity [m^2/s]    B : Skempton's coefficient    nu_u : undrained Poisson's ratio    """    if t <= 0:        return np.zeros_like(z)        arg = z / np.sqrt(4 * c * t)        undrained = B * (1 + nu_u) / (3 * (1 - nu_u)) * p0 * erf(arg)    drained = p0 * erfc(arg)        return undrained + drained# Explore diffusion for a single rainfall eventz = np.linspace(0, 500, 1000)p0 = 1000  # 1 kPa surface load (~0.1 m of water)diffusivities = [0.01, 0.1, 1.0, 10.0]  # m^2/stimes = [1, 7, 30, 90, 365]  # days after eventfig, axes = plt.subplots(2, 2, figsize=(14, 12))# Panel (a): Pressure vs depth at different times, c = 1 m^2/sax = axes[0, 0]c_ref = 1.0  # m^2/sfor t_days in times:    t_sec = t_days * 86400    P = pore_pressure_roeloffs(z, t_sec, p0, c_ref)    ax.plot(P/1e3, z, lw=2, label=f't = {t_days} days')ax.set_xlabel('Pore pressure change [kPa]')ax.set_ylabel('Depth [m]')ax.set_title(f'(a) Pore pressure diffusion, c = {c_ref} m²/s')ax.invert_yaxis()ax.legend()# Panel (b): Effect of diffusivity at t = 30 daysax = axes[0, 1]t_fixed = 30 * 86400for c_val in diffusivities:    P = pore_pressure_roeloffs(z, t_fixed, p0, c_val)    ax.plot(P/1e3, z, lw=2, label=f'c = {c_val} m²/s')ax.set_xlabel('Pore pressure change [kPa]')ax.set_title('(b) Effect of diffusivity at t = 30 days')ax.invert_yaxis()ax.legend()# Panel (c): Undrained vs drained componentsax = axes[1, 0]B, nu_u = 0.8, 0.35t_sec = 30 * 86400c_val = 1.0arg = z / np.sqrt(4 * c_val * t_sec)undrained = B * (1 + nu_u) / (3 * (1 - nu_u)) * p0 * erf(arg)drained = p0 * erfc(arg)total = undrained + drainedax.plot(total/1e3, z, 'k-', lw=2.5, label='Total')ax.plot(undrained/1e3, z, 'b--', lw=2, label='Undrained')ax.plot(drained/1e3, z, 'r--', lw=2, label='Drained')ax.set_xlabel('Pore pressure [kPa]')ax.set_ylabel('Depth [m]')ax.set_title('(c) Undrained vs drained at t = 30 days')ax.invert_yaxis()ax.legend()# Panel (d): Effect of Skempton coefficientax = axes[1, 1]B_values = [0.2, 0.4, 0.6, 0.8, 1.0]for B_val in B_values:    P = pore_pressure_roeloffs(z, t_fixed, p0, c_val, B=B_val)    ax.plot(P/1e3, z, lw=2, label=f'B = {B_val}')ax.set_xlabel('Pore pressure [kPa]')ax.set_title(f"(d) Effect of Skempton's coefficient B")ax.invert_yaxis()ax.legend()fig.suptitle('Poroelastic Pore Pressure Response to Surface Loading\n'             '(Roeloffs, 1988; Clements & Denolle, 2023)', fontsize=14, y=1.02)plt.tight_layout()plt.savefig('fig05_pore_pressure_diffusion.png', bbox_inches='tight')plt.show()

## 2. Groundwater Level Model (Okubo et al., 2024)Okubo et al. (2024, Eq. 4) model the change in groundwater level from precipitation:$$\Delta GWL(t_i) = \sum_{n=0}^{i} \frac{p(t_n)}{\phi} \exp[-\alpha_0 (t_i - t_n)]$$where $\phi$ is porosity and $\alpha_0$ is the exponential decay rate.The dv/v contribution is: $dv/v_{\text{hydro}} = p_1 \cdot \Delta GWL$where $p_1 < 0$ (increased water → decreased velocity via pore pressure).

In [ ]:
# === Groundwater Level Model (Okubo et al. 2024) ===def gwl_model(precipitation, alpha0, phi=0.05, dt=1.0):    """    Compute groundwater level change from precipitation.    Okubo et al. (2024), Eq. 4.        Parameters:    -----------    precipitation : daily precipitation [mm]    alpha0 : exponential decay rate [1/day]    phi : porosity    dt : time step [days]    """    n = len(precipitation)    gwl = np.zeros(n)    for i in range(n):        for j in range(i+1):            gwl[i] += precipitation[j] / phi * np.exp(-alpha0 * (i - j) * dt)    return gwldef gwl_model_fast(precipitation, alpha0, phi=0.05):    """Fast convolution version of GWL model."""    n = len(precipitation)    kernel = np.exp(-alpha0 * np.arange(n))    gwl = np.convolve(precipitation / phi, kernel, mode='full')[:n]    return gwl# Synthetic precipitation (mimicking Parkfield climate)np.random.seed(42)days = np.arange(0, 365*5)# California-like: wet winters, dry summersseasonal = 3.0 * np.maximum(0, np.cos(2*np.pi*(days - 30)/365.25))  # peak in Januarynoise = np.random.exponential(1.0, len(days))precip = seasonal * noise  # mm/dayprecip[precip < 0.1] = 0# Compute GWL for different alpha0alpha0_values = [0.001, 0.005, 0.01, 0.05]  # 1/dayphi = 0.05fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)axes[0].bar(days/365.25, precip, width=1/365.25, color='steelblue', alpha=0.7)axes[0].set_ylabel('Precipitation [mm/day]')axes[0].set_title('(a) Synthetic precipitation (California-like)')for alpha0 in alpha0_values:    gwl = gwl_model_fast(precip, alpha0, phi)    gwl_norm = (gwl - gwl.mean()) / gwl.std()    axes[1].plot(days/365.25, gwl_norm, lw=1.5,                  label=f'$\\alpha_0$ = {alpha0} day$^{{-1}}$ (memory ~ {1/alpha0:.0f} days)')axes[1].set_ylabel('Normalized $\Delta$GWL')axes[1].set_title('(b) Groundwater level response (Okubo et al. 2024, Eq. 4)')axes[1].legend(fontsize=9)# Convert to dv/vp1_values = [-0.05, -0.1, -0.2]  # dv/v per unit GWLalpha0_fixed = 0.01gwl = gwl_model_fast(precip, alpha0_fixed, phi)gwl_anom = gwl - gwl.mean()for p1 in p1_values:    dvv = p1 * gwl_anom / gwl_anom.std() * 0.1  # scale to ~0.1%    axes[2].plot(days/365.25, dvv, lw=1.5, label=f'$p_1$ = {p1}')axes[2].set_ylabel('dv/v [%]')axes[2].set_xlabel('Time [years]')axes[2].set_title('(c) Hydrological dv/v contribution')axes[2].legend()fig.suptitle('Precipitation → Groundwater → dv/v Model\n(Okubo et al. 2024)',              fontsize=14, y=1.02)plt.tight_layout()plt.savefig('fig06_gwl_model.png', bbox_inches='tight')plt.show()

## 3. Surface Loading Scenarios: Ice, Snow, and WaterSurface loads create vertical stress $T_{33}^0$ that competes with pore pressure.Following Fokker et al. (2021), the velocity change has two opposing contributions:$$\frac{d\beta}{\beta} = -\frac{\mu'}{2\mu} u^0 + \frac{\mu' \mp 1}{4\mu} T_{33}^0$$- **Pore pressure** ($u^0$): increases → velocity *decreases* (grain contacts weaken)- **Vertical load** ($T_{33}^0$): increases → velocity *increases* (compaction)We model three scenarios: **ice sheet loading**, **seasonal rainfall**, and **reservoir impoundment**.

In [ ]:
# === Surface Loading Scenarios ===def dvv_surface_load(T33, u0, mu_prime, mu, wave='SV'):    """    dv/v from surface load and pore pressure.    Fokker et al. (2021), Eqs. 9-11.        Parameters:    -----------    T33 : induced vertical stress [Pa] (negative = compression)    u0 : induced pore pressure [Pa]    mu_prime : dimensionless shear modulus derivative    mu : shear modulus [Pa]    wave : 'SV' (vertically polarized) or 'SH' (horizontally polarized)    """    # Pore pressure effect (always opposes confining)    dvv_pore = -mu_prime / (2 * mu) * u0        # Loading effect depends on wave polarization    if wave == 'SV':        dvv_load = -(mu_prime + 1) / (4 * mu) * T33  # SV: propagating vertically    else:        dvv_load = -(mu_prime - 1) / (4 * mu) * T33  # SH        return dvv_pore + dvv_load, dvv_pore, dvv_load# Material parameters (typical shallow sediments, cf. Fokker et al. 2021)mu = 5e8           # shear modulus [Pa] (Vs ~ 450 m/s, rho = 2000)mu_prime = 80      # dmu/dP (dimensionless)# === Scenario 1: Ice Sheet Loading (Glacial) ===ice_thickness = np.linspace(0, 3000, 200)  # mrho_ice = 917  # kg/m^3g = 9.81T33_ice = -rho_ice * g * ice_thickness  # vertical stress (compression = negative)# Assume undrained initially → pore pressure builds up proportionallyB_eff = 0.6  # effective Skempton coefficientu0_ice = -B_eff * T33_ice / 3  # 1/3 because isotropic componentdvv_ice, dvv_pore_ice, dvv_load_ice = dvv_surface_load(T33_ice, u0_ice, mu_prime, mu)# === Scenario 2: Seasonal Rainfall ===days_yr = np.arange(365)rainfall_load = 500 * np.maximum(0, np.cos(2*np.pi*(days_yr-30)/365.25))  # mm equivalentT33_rain = -1000 * g * rainfall_load / 1000  # Pa (water load)u0_rain = 1000 * g * rainfall_load / 1000 * 0.8  # diffused pore pressure (partial)dvv_rain, dvv_pore_rain, dvv_load_rain = dvv_surface_load(T33_rain, u0_rain, mu_prime, mu)# === Scenario 3: Reservoir Impoundment ===water_depth = np.linspace(0, 100, 200)T33_res = -1000 * g * water_depthu0_res = 1000 * g * water_depth * 0.5  # partial drainagedvv_res, dvv_pore_res, dvv_load_res = dvv_surface_load(T33_res, u0_res, mu_prime, mu)fig, axes = plt.subplots(1, 3, figsize=(16, 6))# Ice loadingax = axes[0]ax.plot(ice_thickness, dvv_ice*100, 'k-', lw=2.5, label='Total dv/v')ax.plot(ice_thickness, dvv_pore_ice*100, 'b--', lw=2, label='Pore pressure')ax.plot(ice_thickness, dvv_load_ice*100, 'r--', lw=2, label='Vertical load')ax.set_xlabel('Ice thickness [m]')ax.set_ylabel('dv/v [%]')ax.set_title('(a) Ice Sheet Loading')ax.legend()ax.axhline(0, color='gray', ls='-', alpha=0.5)# Seasonal rainax = axes[1]ax.plot(days_yr, dvv_rain*100, 'k-', lw=2.5, label='Total')ax.plot(days_yr, dvv_pore_rain*100, 'b--', lw=2, label='Pore pressure')ax.plot(days_yr, dvv_load_rain*100, 'r--', lw=2, label='Surface load')ax.set_xlabel('Day of year')ax.set_ylabel('dv/v [%]')ax.set_title('(b) Seasonal Rainfall')ax.legend()ax.axhline(0, color='gray', ls='-', alpha=0.5)# Reservoirax = axes[2]ax.plot(water_depth, dvv_res*100, 'k-', lw=2.5, label='Total')ax.plot(water_depth, dvv_pore_res*100, 'b--', lw=2, label='Pore pressure')ax.plot(water_depth, dvv_load_res*100, 'r--', lw=2, label='Water load')ax.set_xlabel('Reservoir depth [m]')ax.set_ylabel('dv/v [%]')ax.set_title('(c) Reservoir Impoundment')ax.legend()ax.axhline(0, color='gray', ls='-', alpha=0.5)fig.suptitle('Competing Effects: Surface Load vs Pore Pressure\n'             '(Fokker et al. 2021)', fontsize=14, y=1.02)plt.tight_layout()plt.savefig('fig07_surface_loading_scenarios.png', bbox_inches='tight')plt.show()print("Key insight: Pore pressure effect dominates for permeable media (high B),")print("while loading effect dominates for impermeable media (low B).")print(f"Crossover occurs when B ≈ {3*(1-0.35)/(1+0.35):.2f} for nu_u = 0.35")